# Session 02: Machine Learning vs Deep Learning - Hands-On Practice

This notebook breaks down the ML vs DL workflow into digestible steps. We will:
1. Generate synthetic data
2. Explore supervised learning with Scikit-learn
3. Build a deep learning model with PyTorch
4. Understand each step of the training process

Run -> `poetry add torch scikit-learn`

## Part 1: Data Generation

Let's start by generating a synthetic dataset for binary classification.

In [ ]:
from sklearn.datasets import make_classification

# Generate a synthetic classification dataset
X, y = make_classification(
    n_samples=1000,      # Total number of samples
    n_features=10,       # Number of features per sample
    n_classes=2,         # Binary classification (2 classes)
    random_state=42      # For reproducibility
)

print("Dataset shape:")
print(f"  Features (X): {X.shape}")
print(f"  Labels (y): {y.shape}")
print(f"\nClass distribution: {sum(y)} positive samples, {len(y) - sum(y)} negative samples")

## Part 2: Train-Test Split

Split the dataset into training and testing sets to evaluate model generalization.

In [ ]:
from sklearn.model_selection import train_test_split

# Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Train-Test Split:")
print(f"  Training set: {X_train.shape[0]} samples")
print(f"  Testing set: {X_test.shape[0]} samples")

## Part 3: Machine Learning - Random Forest Classifier

Machine Learning typically requires **manual feature engineering**. We train a Random Forest model, which makes decisions based on the input features without learning hierarchical representations.

### Understanding Decision Trees and Random Forests

**Decision Tree:**
A decision tree is a simple model that makes decisions by asking a series of yes/no questions about the features. It splits the data recursively based on feature values to create a tree-like structure. Each split is chosen to best separate the classes.

![Decision Tree Structure - showing hierarchical splits and leaf nodes for binary classification](Decision-Tree-Structure.png)

Example decision path:
```
Is feature_1 > 0.5?
├─ Yes: Is feature_2 > 0.3?
│       ├─ Yes: Predict class 1
│       └─ No: Predict class 0
└─ No: Is feature_3 > 0.7?
        ├─ Yes: Predict class 1
        └─ No: Predict class 0
```

**Random Forest:**
A random forest is an ensemble (collection) of multiple decision trees. Instead of training one tree, we train many trees on random subsets of the data. Each tree votes on the prediction, and the majority vote wins. This voting process reduces overfitting and improves generalization.

![Random Forest Ensemble - showing multiple decision trees voting together for final prediction](Random-Forest-Ensemble.webp)

**Key Differences:**

| Aspect | Decision Tree | Random Forest |
|--------|---------------|---------------|
| **Structure** | Single tree | Multiple trees (ensemble) |
| **Overfitting** | High risk of overfitting | Low (voting reduces overfitting) |
| **Training Data** | Uses all data | Each tree uses random subset |
| **Prediction** | One tree decides | Multiple trees vote |
| **Stability** | Can be unstable with small data changes | More stable and robust |
| **Interpretability** | Easy to visualize and understand | Harder to interpret (many trees) |
| **Accuracy** | Often lower | Usually higher |

In our code below, we use a Random Forest with 100 trees (`n_estimators=100`), which means 100 separate decision trees working together to make predictions.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Step 1: Initialize the classifier
print("Step 1: Initialize the model")
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
print(f"  Model: {rf_model}")

# Step 2: Train (fit) the model
print("\nStep 2: Train the model")
rf_model.fit(X_train, y_train)
print("  Model training complete!")

# Step 3: Make predictions on training data
print("\nStep 3: Predictions on training set")
y_train_pred = rf_model.predict(X_train)
print(f"  First 5 predictions: {y_train_pred[:5]}")
print(f"  First 5 actual labels: {y_train[:5]}")

### Evaluate ML Model

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Predict on test set
y_test_pred = rf_model.predict(X_test)

# Calculate metrics
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)

print("Random Forest Performance:")
print(f"  Training Accuracy: {train_acc:.4f}")
print(f"  Testing Accuracy:  {test_acc:.4f}")
print(f"  Precision:         {precision:.4f}")
print(f"  Recall:            {recall:.4f}")

## Part 4: Deep Learning - Neural Network with PyTorch

Deep Learning learns hierarchical representations directly from raw data. Here we build a neural network that learns non-linear transformations through multiple layers.

### Step 1: Import PyTorch and Prepare Data

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Convert NumPy arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

print("PyTorch Tensors Created:")
print(f"  X_train shape: {X_train_tensor.shape}")
print(f"  y_train shape: {y_train_tensor.shape}")

# Create a DataLoader to batch the data
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print(f"\nDataLoader created with batch_size=32")
print(f"  Number of batches: {len(train_loader)}")

### Step 2: Define the Neural Network Architecture

We use `nn.Sequential()` for a simpler, more concise way to define the network. This approach stacks layers directly without needing a custom class. We output 2 class logits with softmax activation for multi-class scalability.

In [ ]:
# Define the neural network using nn.Sequential with OrderedDict
# This allows us to access layers by name for better debugging and introspection
# Architecture:
# - Input: 10 features
# - Hidden layer 1: 32 neurons + ReLU activation
# - Hidden layer 2: 16 neurons + ReLU activation
# - Output: 2 neurons (for 2 classes: binary classification)

from collections import OrderedDict

model = nn.Sequential(OrderedDict([
    ('fc1', nn.Linear(X_train.shape[1], 32)),  # Input layer: 10 -> 32
    ('relu1', nn.ReLU()),                       # Activation: ReLU
    ('fc2', nn.Linear(32, 16)),                 # Hidden layer: 32 -> 16
    ('relu2', nn.ReLU()),                       # Activation: ReLU
    ('fc3', nn.Linear(16, 2))                   # Output layer: 16 -> 2 (2 classes)
]))

print("Neural Network Architecture:")
print(model)
print("\nNote: Output has 2 neurons for 2 classes, scalable to any number of classes")
print("\nAccessing layers by name:")
print(f"  model['fc1']: {model.fc1}")
print(f"  model['relu1']: {model.relu1}")
print(f"  model['fc3']: {model.fc3}")

### Step 3: Setup Training Components

Before training, we need:
1. **Loss function**: CrossEntropyLoss combines LogSoftmax and NLLLoss for multi-class classification
2. **Optimizer**: Updates weights based on gradients

In [ ]:
# Loss function: CrossEntropyLoss for multi-class classification
# It applies softmax internally, so no need for explicit sigmoid/softmax in the model
criterion = nn.CrossEntropyLoss()
print("Loss Function: CrossEntropyLoss")
print("  Combines softmax activation with negative log likelihood loss")
print("  Ideal for multi-class classification problems")

# Optimizer: Adam with learning rate 0.001
optimizer = optim.Adam(model.parameters(), lr=0.0001)
print("\nOptimizer: Adam")
print("  Updates model weights to minimize loss")
print("  Learning rate: 0.001")

### Step 4: One Training Epoch - Understanding Forward and Backward Pass

Let's go through ONE epoch (one pass through all data) step by step to understand what happens.

In [ ]:
# Switch to training mode
model.train()

# Get the first batch manually to illustrate the steps
batch_X, batch_y = next(iter(train_loader))

# Convert labels to long type (required for CrossEntropyLoss)
batch_y_long = batch_y.long().squeeze()

print("=== BATCH INFORMATION ===")
print(f"Batch shape: {batch_X.shape} (32 samples, 10 features)")
print(f"Labels shape: {batch_y_long.shape}")
print()

# Step 1: FORWARD PASS
print("Step 1: FORWARD PASS")
print("  Call model(batch_X) to get logits (raw scores)")
outputs = model(batch_X)
print(f"  Output shape: {outputs.shape}")
print(f"  Output values (first 3): {outputs[:3].detach().tolist()}")
print(f"  These are logits (scores) for each class before softmax")
print()

# Step 2: CALCULATE LOSS
print("Step 2: CALCULATE LOSS")
print("  Compare logits with actual labels using CrossEntropyLoss")
print("  CrossEntropyLoss applies softmax internally")
loss = criterion(outputs, batch_y_long)
print(f"  Loss value: {loss.item():.4f}")
print(f"  Lower loss = better predictions")
print()

In [ ]:
# Step 3: BACKWARD PASS (Gradient Computation)
print("Step 3: BACKWARD PASS")
print("  Clear old gradients")
optimizer.zero_grad()
print("  Gradients cleared!")
print()

print("  Compute gradients (backpropagation)")
loss.backward()
print("  Gradients computed for every parameter!")
print(f"  Gradient norm for fc1: {model.fc1.weight.grad.norm().item():.6f}")
print(f"  Gradient norm for fc2: {model.fc2.weight.grad.norm().item():.6f}")
print(f"  Gradient norm for fc3: {model.fc3.weight.grad.norm().item():.6f}")
print()

# Step 4: WEIGHT UPDATE
print("Step 4: WEIGHT UPDATE")
print("  Update weights using gradients and learning rate")
print("  new_weight = old_weight - learning_rate * gradient")
optimizer.step()
print("  Weights updated!")
print()

### Step 5: Full Training Loop - Multiple Epochs

In [ ]:
import numpy as np

num_epochs, print_interval = 500, 20
print_indices = set(np.linspace(0, num_epochs-1, print_interval, dtype=int))
training_losses = []

print("Starting full training...")
print()

for epoch in range(num_epochs):
    model.train()  # Set to training mode
    epoch_loss = 0.0
    
    # Loop through all batches
    for batch_X, batch_y in train_loader:
        # Convert labels to long type for CrossEntropyLoss
        batch_y_long = batch_y.long().squeeze()
        
        # 1. Forward pass
        outputs = model(batch_X)
        
        # 2. Calculate loss
        loss = criterion(outputs, batch_y_long)
        
        # 3. Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # 4. Update weights
        optimizer.step()
        
        epoch_loss += loss.item()
    
    # Average loss for the epoch
    avg_loss = epoch_loss / len(train_loader)
    training_losses.append(avg_loss)
    
    if epoch in print_indices:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

print("\nTraining complete!")

### Step 6: Evaluation - Testing the Trained Model

Now we switch to evaluation mode with `model.eval()` and disable gradient computation with `torch.no_grad()` for inference.

In [ ]:
print("=== EVALUATION ===")
print()

# Switch to evaluation mode
model.eval()

# No gradient computation during inference (faster, less memory)
with torch.no_grad():
    # Predictions on training set
    train_outputs = model(X_train_tensor)
    train_predictions = torch.argmax(train_outputs, dim=1).numpy()
    
    # Predictions on test set
    test_outputs = model(X_test_tensor)
    test_predictions = torch.argmax(test_outputs, dim=1).numpy()

# Calculate accuracy
from sklearn.metrics import accuracy_score

train_accuracy = accuracy_score(y_train, train_predictions)
test_accuracy = accuracy_score(y_test, test_predictions)

print("Neural Network Performance (with 2-class softmax output):")
print(f"  Training Accuracy: {train_accuracy:.4f}")
print(f"  Testing Accuracy:  {test_accuracy:.4f}")
print()

print("Comparison with Random Forest:")
print(f"  RF Training Accuracy: {train_acc:.4f}")
print(f"  RF Testing Accuracy:  {test_acc:.4f}")

## Part 5: Visualize 

### Training Loss

Let's visualize how the model's loss decreased over epochs, showing that learning happened.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(training_losses, linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Neural Network Training Loss Over Time', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

print(f"Initial Loss: {training_losses[0]:.4f}")
print(f"Final Loss:   {training_losses[-1]:.4f}")
print(f"Loss Reduction: {(training_losses[0] - training_losses[-1]) / training_losses[0] * 100:.1f}%")

### Decision Boundary

Let's visualize the decision boundary of both models using PCA to reduce the 10D data to 2D.


In [ ]:
from sklearn.decomposition import PCA
import numpy as np

# Reduce to 2D using PCA for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"PCA Explained Variance Ratio: {pca.explained_variance_ratio_}")
print(f"Total Variance Explained: {pca.explained_variance_ratio_.sum():.1%}")

# Create a mesh for decision boundary
h = 0.05  # step size in the mesh
x_min, x_max = X_pca[:, 0].min() - 0.5, X_pca[:, 0].max() + 0.5
y_min, y_max = X_pca[:, 1].min() - 0.5, X_pca[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# Get predictions from both models on the mesh
# We need to transform mesh points back to original 10D space
mesh_points = np.c_[xx.ravel(), yy.ravel()]
mesh_points_original = pca.inverse_transform(mesh_points)

# Random Forest predictions
rf_mesh_pred = rf_model.predict(mesh_points_original)

# PyTorch model predictions
with torch.no_grad():
    mesh_tensor = torch.FloatTensor(mesh_points_original).to(X_train_tensor.device)
    mesh_outputs = model(mesh_tensor)
    dl_mesh_pred = torch.argmax(mesh_outputs, dim=1).cpu().numpy()

# Create side-by-side visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot Random Forest Decision Boundary
rf_mesh_pred_reshaped = rf_mesh_pred.reshape(xx.shape)
axes[0].contourf(xx, yy, rf_mesh_pred_reshaped, levels=np.linspace(0, 1, 3), 
                 cmap='RdYlBu_r', alpha=0.6)
axes[0].scatter(X_test_pca[y_test == 0, 0], X_test_pca[y_test == 0, 1], 
               c='blue', s=20,label='Class 0 (Test)', linewidths=2)
axes[0].scatter(X_test_pca[y_test == 1, 0], X_test_pca[y_test == 1, 1], 
               c='red', s=20, label='Class 1 (Test)', linewidths=2)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=11)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=11)
axes[0].set_title('Random Forest Decision Boundary', fontsize=13, fontweight='bold')
axes[0].legend(loc='best', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Plot Neural Network Decision Boundary
dl_mesh_pred_reshaped = dl_mesh_pred.reshape(xx.shape)
axes[1].contourf(xx, yy, dl_mesh_pred_reshaped, levels=np.linspace(0, 1, 3), 
                 cmap='RdYlBu_r', alpha=0.6)
axes[1].scatter(X_test_pca[y_test == 0, 0], X_test_pca[y_test == 0, 1], 
               c='blue', s=20, label='Class 0 (Test)', linewidths=2)
axes[1].scatter(X_test_pca[y_test == 1, 0], X_test_pca[y_test == 1, 1], 
               c='red', s=20, label='Class 1 (Test)', linewidths=2)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=11)
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=11)
axes[1].set_title('Neural Network Decision Boundary', fontsize=13, fontweight='bold')
axes[1].legend(loc='best', fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nDecision Boundary Visualization Complete!")
print("- Blue regions = Model predicts Class 0")
print("- Red regions = Model predicts Class 1")
print("- Circle markers = Training data")
print("- X markers = Test data")


## Summary: ML vs DL Comparison

### Machine Learning (Random Forest)
- **Feature Engineering**: We passed raw features directly
- **Model Training**: Simple, non-iterative fitting
- **Inference**: One-shot prediction
- **Explainability**: Feature importances are interpretable

### Deep Learning (Neural Network)
- **Representation Learning**: Model learns hierarchical features automatically
- **Model Training**: Iterative optimization over epochs with backpropagation
- **Gradient-Based**: Uses gradients (backward pass) to update weights
- **Scalability**: Better with large datasets and complex patterns

Both achieved high accuracy on this toy problem, but Deep Learning excels with larger, more complex data (images, sequences, text).